# Diffdock NB to start Model Container

In [1]:
import boto3
import json

# Replace with your secret name and region
secret_name = 'NVIDIA_NGC_CREDS_VINCENT'
region_name = 'us-east-1'

# Create a Secrets Manager client
client = boto3.client('secretsmanager', region_name=region_name)

# Retrieve the secret value
response = client.get_secret_value(SecretId=secret_name)
secret = json.loads(response['SecretString'])

# Store the API key in a Python variable
ngc_cli_api_key = secret['NGC_CLI_API_KEY']

# (Optional) Print a success message (avoid printing the actual key)
print("API key retrieved successfully.")

API key retrieved successfully.


In [2]:
%%bash -s "$ngc_cli_api_key"
# Set the API key as an environment variable
export NGC_API_KEY=$1

#test print
#echo "$NGC_API_KEY"

# Authenticate to your ECR registry
aws ecr get-login-password --region us-east-1 | docker login --username AWS --password-stdin 841162696343.dkr.ecr.us-east-1.amazonaws.com

# NVCR login
docker login nvcr.io --username '$oauthtoken' --password $NGC_API_KEY


# V2.0 via nvcr
docker run -d --rm --name diffdock2 \
    --runtime=nvidia -e CUDA_VISIBLE_DEVICES=0 \
    --shm-size=2G \
    --ulimit memlock=-1 \
    --ulimit stack=67108864 \
    -e NGC_API_KEY=$NGC_API_KEY \
    -p 8083:8000 \
    nvcr.io/nim/mit/diffdock:2.0.0




#public.ecr.aws/nvidia/nim:diffdock-1.2.0

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded


WARNING! Using --password via the CLI is insecure. Use --password-stdin.
WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
cb88627ae505d8d3aec0b5eacf6a516247b388241d49d15561c9a68b18ff0488


# check if model runs

In [6]:
!docker ps
# In case you want to stop the container -> !docker stop diffdock

CONTAINER ID   IMAGE                            COMMAND                   CREATED         STATUS         PORTS                                       NAMES
33baeebd6609   nvcr.io/nim/mit/diffdock:2.0.0   "/bin/sh -c 'exec \"$…"   2 minutes ago   Up 2 minutes   0.0.0.0:8083->8000/tcp, :::8083->8000/tcp   diffdock2


# Shut down model

In [3]:
# stop the docker / NIM 
!docker stop diffdock2

diffdock2


In [25]:
!docker ps
# In case you want to stop the container -> !docker stop diffdock

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES
